In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

# DATASETS USED

1. Bengaluru Land Use/Land Cover Dataset: Bengaluru_All_LULC_Points_2024_2025.csv - columns: coordinates, classification of land use/land cover, Classification: 0-urban cover(buildings), 1-vegetation, 2-water, 3-barren land
2. 

# DATA PREPROCESSING (LULC COORDINATES)

In [13]:
def preprocess_lulc(file_path):
    """
    Extracts lat/long from GEE .geo strings.
    """
    df = pd.read_csv(file_path)
    
    def extract_coords(geo_str):
        try:
            # Parse the JSON string from the .geo column
            geo_dict = json.loads(geo_str)
            return pd.Series(geo_dict['coordinates'])
        except:
            return pd.Series([None, None])
            
    # Apply extraction to create separate Longitude and Latitude columns
    df[['longitude', 'latitude']] = df['.geo'].apply(extract_coords)
    df.to_csv('Processed_LULC_Points.csv', index=False)
    print("Success: LULC coordinates extracted.")
    return df

In [19]:
lulc_wardwise_data = pd.read_csv('Bengaluru_All_LULC_Points_2024_2025.csv')

In [14]:
ward_temp = pd.read_csv('Bengaluru_Ward_Temp_Stats.csv')

In [15]:
ward_veg = pd.read_csv('Bengaluru_Ward_Veg_Stats.csv')

In [16]:
ward_water = pd.read_csv('Bengaluru_Ward_Water_Stats.csv')

In [17]:
ward_urban = pd.read_csv('Bengaluru_Ward_urban_Stats.csv')

# WARD ANALYSIS & RANKING

In [23]:
def consolidate_ward_data(temp_file, urban_file, veg_file, water_file):
    """
    Merges ward-level stats and calculates the Environmental Health Score.
    """
    # Load individual CSVs and select relevant columns
    df_temp = pd.read_csv(temp_file)[['Ward_Name', 'Zone', 'mean']].rename(columns={'mean': 'surface_temp'})
    df_urban = pd.read_csv(urban_file)[['Ward_Name', 'mean']].rename(columns={'mean': 'urban_index'})
    df_veg = pd.read_csv(veg_file)[['Ward_Name', 'mean']].rename(columns={'mean': 'veg_index'})
    df_water = pd.read_csv(water_file)[['Ward_Name', 'mean']].rename(columns={'mean': 'water_index'})

    # Merge all datasets on Ward_Name
    master_df = df_temp.merge(df_urban, on='Ward_Name').merge(df_veg, on='Ward_Name').merge(df_water, on='Ward_Name')

    # Normalize parameters to 0-1 range for a fair Health Score
    from sklearn.preprocessing import MinMaxScaler
    
    scaler = MinMaxScaler()
    cols = ['surface_temp', 'urban_index', 'veg_index', 'water_index']
    normed = scaler.fit_transform(master_df[cols])
    df_norm = pd.DataFrame(normed, columns=cols)

    # Health Score Formula: (Vegetation + Water) - (Temperature + Urbanization)
    master_df['health_score'] = (df_norm['veg_index'] + df_norm['water_index']) - \
                                (df_norm['surface_temp'] + df_norm['urban_index'])
    
    # Rank wards: Rank 1 is the healthiest
    master_df['city_rank'] = master_df['health_score'].rank(ascending=False).astype(int)
    master_df = master_df.sort_values('city_rank')
    
    master_df.to_csv('Bengaluru_Ward_Master_Stats.csv', index=False)
    print("Success: Ward health rankings calculated.")
    return master_df

# THE PRESCRIPTIVE ML-BASED TOOL

In [24]:
def get_neighbourhood_report(ward_name, master_df):
    """
    Look up a ward and provide its current status.
    """
    ward_data = master_df[master_df['Ward_Name'].str.lower() == ward_name.lower()]
    if ward_data.empty:
        return "Ward not found."
    return ward_data.to_dict(orient='records')[0]

def get_plot_prescription(plot_size_sqft, num_storeys):
    """
    Calculates architectural interventions based on Bengaluru standards.
    """
    # 1. Minimum Ground Green Cover (15% goal for urban health)
    min_green_cover = plot_size_sqft * 0.15
    
    # 2. Roof Cover Prescription based on density/height
    if num_storeys >= 4:
        roof_rec = "High Density: 60% of terrace must be intensive green roof."
    else:
        roof_rec = "Low/Mid Density: 30% of terrace area for kitchen garden/green roof."
        
    # 3. Rainwater Harvesting (RWH) - 20 Liters per square meter of roof area
    # (1 sqft = 0.0929 sqm)
    rwh_tank_capacity = (plot_size_sqft * 0.0929) * 20 
    
    return {
        "Min Ground Green Cover (sqft)": round(min_green_cover, 2),
        "Roof Target": roof_rec,
        "Est. RWH Tank Size (Liters)": round(rwh_tank_capacity, 2),
        "Vertical Greening": "Required on South/West walls" if num_storeys > 2 else "Recommended"
    }

# MAIN EXECUTION FLOW

In [25]:
if __name__ == "__main__":
    # 1. Define File Paths (adjust as needed)
    LULC_PATH = 'Bengaluru_All_LULC_Points_2024_2025.csv'
    TEMP_CSV = 'Bengaluru_Ward_Temp_Stats.csv'
    URBAN_CSV = 'Bengaluru_Ward_Urban_Stats.csv'
    VEG_CSV = 'Bengaluru_Ward_Veg_Stats.csv'
    WATER_CSV = 'Bengaluru_Ward_Water_Stats.csv'

    # 2. Preprocess and Rank
    processed_lulc = preprocess_lulc(LULC_PATH)
    master_stats = consolidate_ward_data(TEMP_CSV, URBAN_CSV, VEG_CSV, WATER_CSV)

    # 3. Example Usage: User selects a Ward and enters Plot Details
    my_ward = "Halsoor"
    plot_size = 1200 # 30x40 site
    storeys = 3

    print("\n--- NEIGHBOURHOOD REPORT ---")
    report = get_neighbourhood_report(my_ward, master_stats)
    print(f"Ward: {report['Ward_Name']} | City Rank: {report['city_rank']} / 198")
    print(f"Health Score: {round(report['health_score'], 2)}")

    print("\n--- PLOT PRESCRIPTION ---")
    prescription = get_plot_prescription(plot_size, storeys)
    for key, value in prescription.items():
        print(f"{key}: {value}")

Success: LULC coordinates extracted.
Success: Ward health rankings calculated.

--- NEIGHBOURHOOD REPORT ---
Ward: Halsoor | City Rank: 1 / 198
Health Score: 1.24

--- PLOT PRESCRIPTION ---
Min Ground Green Cover (sqft): 180.0
Roof Target: Low/Mid Density: 30% of terrace area for kitchen garden/green roof.
Est. RWH Tank Size (Liters): 2229.6
Vertical Greening: Required on South/West walls


In [3]:
Prominent_Trees = pd.read_csv("Annexure3_Prominent_Trees.csv")

In [15]:
Prominent_Trees.head()

,Common_Name,Family,Description,Flowering,Native,Location
0,Australian wattle,Fabaceae,A medium size evergreen tree. Leaves alternate...,July- October,Australia,"Anajana Nagar, Yeshwanthpur, Vijay"
1,Butterfly tree,Fabaceae,A medium-sized tree up to 6.5 m high. Leaves a...,NaN,"India, Burma, Vietnam","Malleshwaram, Mahalakshmipuram,"
2,Red silk-cotton tree,Bombacaceae,A tall native with straight trunk that is cove...,NaN,India,"Malleshwaram, M.G Road, Sadashiva Nagar."
3,Popcorn bush cedar,Fabaceae,A medium sized tree with dense canopy. Leaves ...,NaN,Tropical Southeast Asia,"Malleshwaram, Sanjay Nagar, Sankey Road, M.G"
4,Coconut palm,Arecaceae,A monoecious palm with regular leaf scars. Lea...,NaN,Indo-Pacific,"Rajaji Nagar, Mahalakshmi layout, Rajajeshwari"


In [17]:
Prominent_Trees[Prominent_Trees['Family']=='Casuarinaceae']

,Common_Name,Family,Description,Flowering,Native,Location
14,She- oak,Casuarinaceae,"Dioecious tree, up to 15 m high. Stem cylindri...",NaN,"Malaysia, S. Asia, Australia","Yelahanka, Ramanagaram, Malleshwaram."


In [21]:
Prominent_Trees['Family'].unique()

array(['Fabaceae', 'Bombacaceae', 'Arecaceae', nan, 'Annonaceae',
       'Bignoniaceae', 'Casuarinaceae', 'Verbenaceae', 'Proteaceace',
       'Meliaceae', 'Magnoliaceae', 'Rutaceae', 'Myrtaceae',
       'Anacardiaceae', 'Malvaceae', 'Solanaceae', 'Lythraceae'],
      dtype=object)

In [6]:
Prominent_Trees.isnull().sum()

Common_Name     0
Family          1
Description     0
Flowering      22
Native          0
Location        0
dtype: int64

In [8]:
bengaluru_trees = pd.read_csv("Bengaluru_Tree_Palette.csv")

In [9]:
bengaluru_trees

,Sl.\nNo,Botanical Name,Family,Location
0,1,Casuarina equisetifolia,Casuarinaceae,"Yelahanka, Ramanagaram"
1,2,Salix tetrasperma,Salicaceae,Ramanagaram
2,3,Holoptelea integrifolia,Ulmaceae,"Ramanagaram, Jayachanarajendra Technological I..."
3,4,Trema orientalis,Ulmaceae,"Savandurga, Ramanagaram"
4,5,Broussonetia papyrifera,Moraceae,"SVR 503, 2740 Bangalore"
...,...,...,...,...
112,113,Ligustrum roxburghii,Oleaceae,NaN
113,114,Plumeria rubra L. forma acuminata,Apocynaceae,NaN
114,115,Alstonia scholaris,Apocynaceae,NaN
115,116,Wrightia tinctoria,Apocynaceae,"Ramanaragaram, Savandurga, Bannerghatta"
